In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import pickle
from tqdm import tqdm

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
members = range(1,30)
for i in tqdm(members):
    sm = load_samap('sm_Allen_Full_dr_ncbi_joined_cleaned_07172026_'+str(i)+'.pkl')
    org = 'dr'
    ref = 'mg'
    eq_subclass_lc = pd.read_csv('eq_subclass_lc_dr_cleaned_07202026.csv', index_col = 'Unnamed: 0')
    sm.sams[org].adata.obs['eq_subclass_lc'] = eq_subclass_lc
    
    keys = {org:'eq_subclass_lc',ref:'subclass_id_label'}
    D,MappingTable = get_mapping_scores(sm,keys)

    lim_MappingTable = MappingTable.filter(like=org + '_')
    lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]
        
    mapping_dict = {}
    for item in lim_MappingTable:
        len_item = len(sm.sams[org].adata[sm.sams[org].adata.obs['eq_subclass_lc'] == int(item[3:])])
        if len_item > 25:
            mapping_dict[item] = str(lim_MappingTable[item].idxmax())
        else:
            mapping_dict[item] = ref + '_Unlabeled'
            
    new_mapping = []
    for item in sm.sams[org].adata.obs['eq_subclass_lc']:
        new_mapping.append(mapping_dict[org + '_' + str(item)][3:])
        
    df = pd.DataFrame(data = new_mapping, index = sm.sams[org].adata.obs_names, columns = [str(i)])
    df.to_csv('DR_MG_mapping_cleaned_07172026_'+str(i)+'.csv')

100%|██████████████████████████████████████████| 29/29 [58:37<00:00, 121.30s/it]


In [4]:
fn = 'sm_Allen_Full_dr_cleaned_07172026_ncbi.pkl'

In [5]:
org = 'dr'
ref = 'mg'
sm = load_samap(fn)
eq_subclass_lc = pd.read_csv('eq_subclass_lc_dr_cleaned_07202026.csv', index_col = 'Unnamed: 0')
sm.sams[org].adata.obs['eq_subclass_lc'] = eq_subclass_lc

keys = {org:'eq_subclass_lc',ref:'subclass_id_label'}
D,MappingTable = get_mapping_scores(sm,keys)

lim_MappingTable = MappingTable.filter(like=org + '_')
lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

mapping_dict = {}
for item in lim_MappingTable:
    len_item = len(sm.sams[org].adata[sm.sams[org].adata.obs['eq_subclass_lc'] == int(item[3:])])
    if len_item > 25:
        mapping_dict[item] = str(lim_MappingTable[item].idxmax())
    else:
        mapping_dict[item] = ref + '_Unlabeled'

new_mapping = []
for item in sm.sams[org].adata.obs['eq_subclass_lc']:
    new_mapping.append(mapping_dict[org + '_' + str(item)][3:])

df = pd.DataFrame(data = new_mapping, index = sm.sams[org].adata.obs_names, columns = [str(0)])
df.to_csv('DR_MG_mapping_cleaned_07172026_0.csv')